In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    desc,
    row_number,
    sha2,
    when,
    lower,
    sum as sum_,
    rtrim,
    max as max_,
    initcap,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyID string"
        #     ",SourceSystem string"
        #     ",PortfolioKey string"
        #     ",PortfolioName string"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId string"
        #     ",CDI string"
        #     ",RepresentedByPartyKey string"
        #     ",RepresentedByPartyName string"
        #     ",RepresentedByPartyType string"
        #     ",PartyKey string"
        #     ",PartyRepresentKey string"
        #     ",PartyTypeCode string"
        #     ",PartyType string"
        #     ",PartyStatusCode string"
        #     ",PartyStatus string"
        #     ",PartyStatusDate date"
        #     ",L3869StatusCode string"
        #     ",L3869Status string"
        #     ",L3869CourtDate timestamp"
        #     ",OCWStatus string"
        #     ",DeceasedFlag boolean"
        #     ",L3869InterbankingStatusCode int"
        #     ",L3869InterBankingStatus string"
        #     ",Watchlist string"
        #     ",FirstNameGR string"
        #     ",FirstNameENG string"
        #     ",LastNameGR string"
        #     ",LastNameENG string"
        #     ",FirstFatherNameGR string"
        #     ",FirstFatherNameENG string"
        #     ",LastFatherNameGR string"
        #     ",LastFatherNameENG string"
        #     ",FirstMotherNameGR string"
        #     ",LastMotherNameGR string"
        #     ",LastMotherNameENG string"
        #     ",LegalNameGR string"
        #     ",LegalNameENG string"
        #     ",PrimaryCommercialGR string"
        #     ",PrimaryCommercialENG string"
        #     ",SalutationTypeCode string"
        #     ",SalutationType string"
        #     ",GenderCode string"
        #     ",Gender string"
        #     ",MaritalStatusCode string"
        #     ",MaritalStatus string"
        #     ",CitizenshipCode string"
        #     ",Citizenship string"
        #     ",NationalityCode string"
        #     ",Nationality string"
        #     ",OccupationCode string"
        #     ",Occupation string"
        #     ",BirthDate string"
        #     ",BirthPlace string"
        #     ",DeathDate date"
        #     ",CommunicationLanguageCode int"
        #     ",CommunicationLanguage string"
        #     ",TaxRegistrationNumber string"
        #     ",TaxResidenceCountry string"
        #     ",OrganizationOriginCountry string"
        #     ",PassportNumber string"
        #     ",PassportIssuingAuthority string"
        #     ",PassportIssueDate date"
        #     ",PassportExpiredDate date"
        #     ",NationalNumber string"
        #     ",NationalNumberIssuingAuthority string"
        #     ",NationalNumberIssueDate date"
        #     ",NationalNumberExpiredDate date"
        #     ",MilitaryNumber string"
        #     ",MilitaryAuthority string"
        #     ",MilitaryIssueDate date"
        #     ",MilitaryExpiredDate date"
        #     ",SocialSecurityNumber string"
        #     ",ChildrenCertification string"
        #     ",ChildrenCertificationIssuingAuthority string"
        #     ",EnterpriseStartDate string"
        #     ",EnterpriseEndDate date"
        #     ",EconomicActivityCode int"
        #     ",EconomicActivity string"
        #     ",BranchActivityCode int"
        #     ",BranchActivity string"
        #     ",NumberOfChildren string"
        #     ",GeneralCommercialRegister string"
        #     ",NumberofEmployees string"
        #     ",TotalIncome string"
        #     ",BalanceSheetTotalAmount string"
        #     ",AnnualTurnoverAmount string"
        #     ",GroupCode string"
        #     ",GroupName string"
        #     ",GroupCodePortfolio int"
        #     ",PartyBankruptcyStage int"
        #     ",CovidFlag string"
        #     ",CovidCategoryCode int"
        #     ",CovidCategory string"
        #     ",AffectedCategoryCode int"
        #     ",AffectedCategory string"
        #     ",MonitoringUnitCode string"
        #     ",MonitoringUnit string"
        #     ",CreditRatingTypeCode int"
        #     ",CreditRatingType string"
        #     ",EducationLevelCode int"
        #     ",EducationLevel string"
        #     ",BalanceCategoryCode string"
        #     ",BalanceCategory string"
        #     ",ExtraSegmentCode string"
        #     ",ExtraSegment string"
        #     ",SegmentCode string"
        #     ",Segment string"
        #     ",SubSegmentCode string"
        #     ",SubSegment string"
        #     ",LegalFormCode int"
        #     ",LegalForm string"
        #     ",IsWholeshaleFlag boolean"
        #     ",EmploymentStatusCode int"
        #     ",EmploymentStatus string"
        #     ",EmploymentStatusDate date"
        #     ",CreatedBy string"
        #     ",CreatedByDate timestamp"
        #     ",LastUpdatedBy string"
        #     ",LastUpdatedByDate timestamp"
        #     ",ResubmissionStatus string"
        #     ",N4605Flag int"
        #     ",ConnectedBorrowerFlag boolean"
        #     ",StaffFlag string"
        #     ",L3869BorrowerHasOver50KBalance boolean"
        #     ",TargetLegalActionCode int"
        #     ",TargetLegalAction string"
        #     ",SubServicer string"
        #     ",RiskDescription string"
        #     ",MobileTel string"
        #     ",MobileTel2 string"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["businessdate"]

        curated_table_path = f"{curated_zone}.Party"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== customer_relation ==================

        customer_relation_extra_filters = None

        if customer_relation_extra_filters is not None:
            customer_relation_extra_filters.append(date_filter)
            customer_relation_final_filters = " AND ".join(
                customer_relation_extra_filters
            )
        else:
            customer_relation_final_filters = date_filter

            customer_relation_selected_fields = [
                "sec_cust_id",
                "relation",
                "FST_CUST_ID",
                "FST_NAME",
                "businessdate",
            ]

            customer_relation = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "profits_customer_relation",
                customer_relation_final_filters,
                customer_relation_selected_fields,
            )

        # ================== user_defined_fields ==================

        user_defined_fields_extra_filters = None

        if user_defined_fields_extra_filters is not None:
            user_defined_fields_extra_filters.append(date_filter)
            user_defined_fields_final_filters = " AND ".join(
                user_defined_fields_extra_filters
            )
        else:
            user_defined_fields_final_filters = date_filter

            user_defined_fields_selected_fields = [
                "field_value",
                "UD_ACCOUNT_NUMBER",
                "PFG_TAG_SET_CODE",
                "PFG_TAG",
            ]

            user_defined_fields = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "profits_user_defined_fields",
                user_defined_fields_final_filters,
                user_defined_fields_selected_fields,
            )

        # ================== customer_category ==================

        customer_category_extra_filters = None

        if customer_category_extra_filters is not None:
            customer_category_extra_filters.append(date_filter)
            customer_category_final_filters = " AND ".join(
                customer_category_extra_filters
            )
        else:
            customer_category_final_filters = date_filter

        customer_category_selected_fields = [
            "serial_num",
            "CUST_ID",
            "param_type",
            "businessdate",
        ]

        customer_category = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer_category",
            customer_category_final_filters,
            customer_category_selected_fields,
        )

        # ================================ params ================================
        params_extra_filters = None

        if params_extra_filters is not None:
            params_extra_filters.append(date_filter)
            params_final_filters = " AND ".join(params_extra_filters)
        else:
            params_final_filters = date_filter

        params_selected_fields = ["description", "serial_num", "param_txt_1"]

        params = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_params",
            params_final_filters,
            params_selected_fields,
        )

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "CUST_ID",
            "unit_code",
            "businessdate",
            "CDI",
            "CUST_TYPE",
            "CUST_STATUS",
            "STATUS_DATE",
            "FIRST_NAME",
            "LATIN_FIRSTNAME",
            "SURNAME",
            "LATIN_SURNAME",
            "FATHER_NAME",
            "MOTHER_NAME",
            "NAME_STANDARD",
            "TITLE",
            "SEX",
            "DATE_OF_BIRTH",
            "TAX_NUMBER",
            "ID_NO",
            "ISSUE_DATE",
            "EXPIRE_DATE",
            "gd_CRRAT",
            "gd_BHFMR",
            "MOBILE_TEL",
            "MOBILE_TEL2",
            "businessdate",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer",
            customer_final_filters,
            customer_selected_fields,
        )
        # ================== group_member ==================

        group_member_extra_filters = None

        if group_member_extra_filters is not None:
            group_member_extra_filters.append(date_filter)
            group_member_final_filters = " AND ".join(group_member_extra_filters)
        else:
            group_member_final_filters = date_filter

        group_member_selected_fields = [
            "CUST_ID",
            "AB_GROUP",
            "Group_Description",
            "Profits_Group_id",
            "BusinessDate",
        ]

        group_member = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_group_member",
            group_member_final_filters,
            group_member_selected_fields,
        )

        # ================== other_id ==================

        other_id_extra_filters = None

        if other_id_extra_filters is not None:
            other_id_extra_filters.append(date_filter)
            other_id_final_filters = " AND ".join(other_id_extra_filters)
        else:
            other_id_final_filters = date_filter

        other_id_selected_fields = [
            "FK_CUSTOMERCUST_ID",
            "FKGD_HAS_TYPE",
            "ID_NO",
            "BusinessDate",
        ]

        other_id = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_other_id",
            other_id_final_filters,
            other_id_selected_fields,
        )

        # ================== generic_detail ==================

        generic_detail_extra_filters = None

        if generic_detail_extra_filters is not None:
            generic_detail_extra_filters.append(date_filter)
            generic_detail_final_filters = " AND ".join(generic_detail_extra_filters)
        else:
            generic_detail_final_filters = date_filter

        generic_detail_selected_fields = [
            "SERIAL_NUM",
            "PARAMETER_TYPE",
            "DESCRIPTION",
            "BusinessDate",
        ]

        generic_detail = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_generic_detail",
            generic_detail_final_filters,
            generic_detail_selected_fields,
        )

        # ================== cust_address ==================

        cust_address_extra_filters = None

        if cust_address_extra_filters is not None:
            cust_address_extra_filters.append(date_filter)
            cust_address_final_filters = " AND ".join(cust_address_extra_filters)
        else:
            cust_address_final_filters = date_filter

        cust_address_selected_fields = [
            "FK_CUSTOMERCUST_ID",
            "FKGD_HAS_COUNTRY",
            "BusinessDate",
            "COMMUNICATION_ADDR"
        ]

        cust_address = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_cust_address",
            cust_address_final_filters,
            cust_address_selected_fields,
        )

         # ================== aml_prod customers_export ==================

        customers_export_table = f"`{env_prefix}-aml`.aml_prod.customers_export"

        export_date = (
            spark.read.table(customers_export_table)
            .filter(col("export_date") <= run_date)
            .agg(max_("export_date"))
            .collect()[0][0]
        )

        customers_export = (
            spark.read.table(customers_export_table)
            .filter(col("export_date") == export_date)
            .select("TaxRegistrationNumber", "FinalRiskClass")
        ).distinct()

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "portfolioname",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== party_profits_df ==================

        maritalstatuscode_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "FALST"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΟΙΚΟΓΕΝΕΙΑΚΗ ΚΑΤΑΣΤΑΣΗ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_maritalstatuscode_code"),
                customer_category["serial_num"].alias("MaritalStatusCode"),
            )
        )

        citizenship_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "CITIZ"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΥΠΗΚΟΟΤΗΤΑ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_citizenship_code"),
                customer_category["serial_num"].alias("CitizenshipCode"),
                params["description"].alias("Citizenship"),
            )
        )
        nationalitycode_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "NATIO"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΕΘΝΙΚΟΤΗΤΑ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_nationalitycode_code"),
                customer_category["serial_num"].alias("NationalityCode"),
            )
        )
        occupationcode_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "PROFF"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΕΠΑΓΓΕΛΜΑ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_occupationcode_code"),
                customer_category["serial_num"].alias("OccupationCode"),
            )
        )
        communicationlanguage_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "COMLA"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΓΛΩΣΣΑ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_communicationlanguage_code"),
                customer_category["serial_num"].alias("CommunicationLanguageCode"),
                params["description"].alias("CommunicationLanguage"),
            )
            .distinct()
            .filter(
                (col("CUST_ID_communicationlanguage_code").isNotNull())
                & (col("CommunicationLanguage").isNotNull())
            )
        )
        economicactivitycode_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "CCODE"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΚΛΑΔΟΣ ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_economicactivitycode_code"),
                customer_category["serial_num"].alias("EconomicActivityCode"),
            )
        )
        branchactivitycode_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "CCODE"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΚΛΑΔΟΣ ΟΙΚΟΝΟΜΙΑΣ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_branchactivitycode_code"),
                customer_category["serial_num"].alias("BranchActivityCode"),
            )
        )
        educationlevel_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "EDULV"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΕΠΙΠΕΔΟ ΜΟΡΦΩΣΗΣ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_educationlevel_code"),
                customer_category["serial_num"].alias("EducationLevelCode"),
                params["description"].alias("EducationLevel"),
            )
        )
        educationlevelcode_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "EDULV"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΕΠΑΓΓΕΛΜΑ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_educationlevelcode_code"),
                customer_category["serial_num"].alias("EducationLevelCode"),
            )
        )
        legalformcode_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "LEGAL"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΝΟΜΙΚΗ ΜΟΡΦΗ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_legalformcode_code"),
                customer_category["serial_num"].alias("LegalFormCode"),
                params["description"].alias("LegalForm"),
            )
        )
        employmentstatus_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "NATIO"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΕΠΑΓΓΕΛΜΑΤΙΚΗ ΚΑΤΑΣΤΑΣΗ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_employmentstatus_code"),
                customer_category["serial_num"].alias("EmploymentStatusCode"),
                params["description"].alias("EmploymentStatus"),
            )
        )
        maritalstatus_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "FALST"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΟΙΚΟΓΕΝΕΙΑΚΗ ΚΑΤΑΣΤΑΣΗ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_maritalstatus_code"),
                params["description"].alias("MaritalStatus"),
            )
        )
        nationality_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "NATIO"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΕΘΝΙΚΟΤΗΤΑ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_nationality_code"),
                params["description"].alias("Nationality"),
            )
        )
        occupation_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "PROFF"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΕΠΑΓΓΕΛΜΑ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_occupation_code"),
                customer_category["serial_num"].alias("OccupationCode"),
                params["description"].alias("Occupation"),
            )
        )
        economicactivity_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "CCODE"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΚΛΑΔΟΣ ΔΡΑΣΤΗΡΙΟΤΗΤΑΣ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_economicactivity_code"),
                customer_category["serial_num"].alias("EconomicActivityCode"),
                params["description"].alias("EconomicActivity"),
            )
        )
        BranchActivity_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "CCODE"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΚΛΑΔΟΣ ΟΙΚΟΝΟΜΙΑΣ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_BranchActivity_code"),
                params["description"].alias("BranchActivity"),
                customer_category["serial_num"].alias("BranchActivityCode"),
            )
        )
        legalform_code = (
            customer.join(
                customer_category,
                (customer["CUST_ID"] == customer_category["CUST_ID"])
                & (customer_category["param_type"] == "LEGAL"),
                "left",
            )
            .join(
                params, customer_category["serial_num"] == params["serial_num"], "left"
            )
            .where(params["param_txt_1"] == "ΝΟΜΙΚΗ ΜΟΡΦΗ")
            .select(
                customer["CUST_ID"].alias("CUST_ID_legalform_code"),
                params["description"].alias("LegalForm"),
            )
        )
        
        organizationorigincountry_code = (
            customer.join(
                other_id,
                (other_id["FK_CUSTOMERCUST_ID"] == customer["CUST_ID"])
                & (other_id["FKGD_HAS_TYPE"] == 13),
                "left",
            )
            .join(
                cust_address, cust_address["FK_CUSTOMERCUST_ID"] == customer["CUST_ID"], 
                "left",
            )
            .join(
                generic_detail, 
                (cust_address["FKGD_HAS_COUNTRY"] == generic_detail["SERIAL_NUM"])
                & (generic_detail["PARAMETER_TYPE"] == "CNTRY"), 
                "left",
            )
            .withColumn(
                "is_default", (col("DESCRIPTION") == "DEFAULT").cast("int")
            )
            .withColumn(
                "rank",
                row_number().over(
                    Window.partitionBy(customer["CUST_ID"]).orderBy(col("is_default"), generic_detail["DESCRIPTION"])
                    )
            )
            .where((rtrim(other_id["ID_NO"]) != '')
                   & (customer["CUST_TYPE"] == 2)
                   & (cust_address["COMMUNICATION_ADDR"] == 1)
                   )
            .select(
                customer["CUST_ID"].alias("CUST_ID_organizationcountrycode"),
                when(generic_detail["DESCRIPTION"] == "DEFAULT", lit(None)
                     ).otherwise(generic_detail["DESCRIPTION"]).alias("OrganizationOriginCountry")
                ).distinct()
        )


        customer_primary_key_fields = [
            customer.CUST_ID,
            customer_relation.FST_CUST_ID,
            lit("profits"),
        ]

        party_profits_df = (
            customer.join(
                citizenship_code,
                customer.CUST_ID == citizenship_code.CUST_ID_citizenship_code,
                "left",
            )
            .join(
                maritalstatuscode_code,
                customer.CUST_ID
                == maritalstatuscode_code.CUST_ID_maritalstatuscode_code,
                "left",
            )
            .join(
                nationalitycode_code,
                customer.CUST_ID == nationalitycode_code.CUST_ID_nationalitycode_code,
                "left",
            )
            .join(
                occupationcode_code,
                customer.CUST_ID == occupationcode_code.CUST_ID_occupationcode_code,
                "left",
            )
            .join(
                communicationlanguage_code,
                customer.CUST_ID
                == communicationlanguage_code.CUST_ID_communicationlanguage_code,
                "left",
            )
            .join(
                economicactivitycode_code,
                customer.CUST_ID
                == economicactivitycode_code.CUST_ID_economicactivitycode_code,
                "left",
            )
            .join(
                branchactivitycode_code,
                customer.CUST_ID
                == branchactivitycode_code.CUST_ID_branchactivitycode_code,
                "left",
            )
            .join(
                educationlevel_code,
                customer.CUST_ID == educationlevel_code.CUST_ID_educationlevel_code,
                "left",
            )
            .join(
                educationlevelcode_code,
                customer.CUST_ID
                == educationlevelcode_code.CUST_ID_educationlevelcode_code,
                "left",
            )
            .join(
                legalformcode_code,
                customer.CUST_ID == legalformcode_code.CUST_ID_legalformcode_code,
                "left",
            )
            .join(
                employmentstatus_code,
                customer.CUST_ID == employmentstatus_code.CUST_ID_employmentstatus_code,
                "left",
            )
            .join(
                maritalstatus_code,
                customer.CUST_ID == maritalstatus_code.CUST_ID_maritalstatus_code,
                "left",
            )
            .join(
                nationality_code,
                customer.CUST_ID == nationality_code.CUST_ID_nationality_code,
                "left",
            )
            .join(
                occupation_code,
                customer.CUST_ID == occupation_code.CUST_ID_occupation_code,
                "left",
            )
            .join(
                economicactivity_code,
                customer.CUST_ID == economicactivity_code.CUST_ID_economicactivity_code,
                "left",
            )
            .join(
                BranchActivity_code,
                customer.CUST_ID == BranchActivity_code.CUST_ID_BranchActivity_code,
                "left",
            )
            .join(
                legalform_code,
                customer.CUST_ID == legalform_code.CUST_ID_legalform_code,
                "left",
            )
            .join(
                customer_relation,
                customer.CUST_ID == customer_relation.sec_cust_id,
                "left",
            )
            .join(
                organizationorigincountry_code,
                customer.CUST_ID == organizationorigincountry_code.CUST_ID_organizationcountrycode,
                "left",
            )
            .join(
                portfolio,
                customer.unit_code == portfolio.unit_code_profits,
                "left",
            )
            .join(
                user_defined_fields,
                (customer["CUST_ID"] == user_defined_fields["UD_ACCOUNT_NUMBER"])
                & (user_defined_fields["PFG_TAG_SET_CODE"] == "OCW STATUS")
                & (user_defined_fields["PFG_TAG"] == "PFTAG401"),
                "left",
            )
            .join(
                group_member,
                (customer.CUST_ID == group_member.CUST_ID)
                & (customer.businessdate == group_member.BusinessDate),
                "left",
            )
            .join(
                customers_export,
                (customer.TAX_NUMBER == customers_export.TaxRegistrationNumber),
                "left",
            ) 
            .withColumn(
                "PartyType",
                when(customer.CUST_TYPE == 1, "Individual")
                .when(customer.CUST_TYPE == 2, "Organization")
                .cast("string"),
            )
            .withColumn(
                "Gender",
                when(customer.SEX == "F", "Female")
                .when(customer.SEX == "M", "Male")
                .cast("string"),
            )
            .withColumn(
                "BirthDate",
                when(customer.DATE_OF_BIRTH.isNotNull(), customer.DATE_OF_BIRTH)
                .otherwise("")
                .cast("timestamp"),
            )
            .withColumn("PassportIssuingAuthority", lit(None).cast("string"))
            .withColumn("MilitaryID", lit(None).cast("string"))
            .withColumn("MilitaryAuthority", lit(None).cast("string"))
            .withColumn("ChildrenCertification", lit(None).cast("string"))
            .withColumn(
                "ChildrenCertificationIssuingAuthority", lit(None).cast("string")
            )
            .withColumn("TotalIncome", lit(None).cast("string"))
            .withColumn("PartyBankruptcyStage", lit(None).cast("int"))
            .withColumn("CovidFlag", lit(None).cast("int"))
            .withColumn("CovidCategoryCode", lit(None).cast("int"))
            .withColumn("BalanceCategoryCode", lit(None).cast("int"))
            .withColumn("ExtraSegmentCode", lit(None).cast("string"))
            .withColumn("SegmentCode", lit(None).cast("string"))
            .withColumn("SubSegmentCode", lit(None).cast("string"))
            .withColumn("PartyKeyART", lit(None).cast("string"))
            .withColumn("LastMotherNameGR", lit(None).cast("string"))
            .withColumn("NationalIDIssuingAuthority", lit(None).cast("string"))
            .withColumn("SocialSecurityNumber", lit(None).cast("string"))
            .withColumn("EnterpriseStartDate", lit(None).cast("date"))
            .withColumn(
                "PartyID", sha2(concat_ws(";", *customer_primary_key_fields), 256)
            )
            # .withColumn(
            #     "RepresentedByPartyKey",
            #     when(
            #         customer_relation.sec_cust_id.isNotNull(),
            #         customer_relation.sec_cust_id,
            #     )
            #     .otherwise("")
            #     .cast("string"),
            # )
            .withColumn(
                "NationalNumberIssueDate",
                when(
                    customer.ISSUE_DATE < "1900-01-01",
                    lit(None),
                ).otherwise(customer.ISSUE_DATE),
            )
            .withColumn(
                "DeceasedFlag", when((customer.CUST_STATUS == 4), True).otherwise(False)
            )
            .withColumn(
                "PartyRepresentKey",
                concat_ws(
                    ";",
                    customer.CUST_ID,
                    customer_relation.FST_CUST_ID,
                ),
            )
            .withColumn(
                "PartyStatus",
                when(
                    (customer.CUST_STATUS == 1) | (customer.CUST_STATUS == 2),
                    "Active",
                )
                .when(
                    (customer.CUST_STATUS == 3),
                    "Dormant",
                )
                .when(
                    (customer.CUST_STATUS == 4),
                    "Deceased",
                )
                .when(
                    (customer.CUST_STATUS == 5),
                    "Closed",
                ),
            )
            .withColumn(
                "RiskDescription",
                initcap(customers_export.FinalRiskClass)
                .cast("string"),
            )
        )

        citizenship_code.unpersist()
        economicactivity_code.unpersist()
        BranchActivity_code.unpersist()
        legalform_code.unpersist()
        customer_relation.unpersist()
        nationalitycode_code.unpersist()
        occupationcode_code.unpersist()
        communicationlanguage_code.unpersist()
        economicactivitycode_code.unpersist()
        branchactivitycode_code.unpersist()
        educationlevel_code.unpersist()
        educationlevelcode_code.unpersist()
        legalformcode_code.unpersist()
        employmentstatus_code.unpersist()
        maritalstatus_code.unpersist()
        nationality_code.unpersist()

        # ================== Selection of Final Columns ==================

        party_profits_df_final = party_profits_df.select(
            party_profits_df.PartyID.alias("PartyID"),
            lit("profits").cast("string").alias("SourceSystem"),
            customer.unit_code.cast("string").alias("PortfolioKey"),
            portfolio.portfolioname.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            customer.CDI.cast("string").alias("CDI"),
            customer_relation.FST_CUST_ID.cast("string").alias("RepresentedByPartyKey"),
            customer_relation.FST_NAME.cast("string").alias("RepresentedByPartyName"),
            customer_relation.relation.cast("string").alias("RepresentedByPartyType"),
            party_profits_df.PartyRepresentKey.cast("string").alias(
                "PartyRepresentKey"
            ),
            customer.CUST_ID.cast("string").alias("PartyKey"),
            customer.CUST_TYPE.cast("string").alias("PartyTypeCode"),
            party_profits_df.PartyType.cast("string").alias("PartyType"),
            customer.CUST_STATUS.cast("string").alias("PartyStatusCode"),
            party_profits_df.PartyStatus.cast("string").alias("PartyStatus"),
            customer.STATUS_DATE.cast("date").alias("PartyStatusDate"),
            lit(None).alias("L3869StatusCode"),
            lit(None).alias("L3869Status"),
            lit(None).alias("L3869CourtDate"),
            user_defined_fields.field_value.alias("OCWStatus"),
            party_profits_df.DeceasedFlag.alias("DeceasedFlag"),
            lit(None).alias("L3869InterbankingStatusCode"),
            lit(None).alias("L3869InterBankingStatus"),
            lit(None).alias("Watchlist"),
            customer.FIRST_NAME.cast("string").alias("FirstNameGR"),
            customer.LATIN_FIRSTNAME.cast("string").alias("FirstNameENG"),
            customer.SURNAME.cast("string").alias("LastNameGR"),
            customer.LATIN_SURNAME.cast("string").alias("LastNameENG"),
            customer.FATHER_NAME.cast("string").alias("FirstFatherNameGR"),
            lit(None).cast("string").alias("FirstFatherNameENG"),
            lit(None).cast("string").alias("LastFatherNameGR"),
            lit(None).cast("string").alias("LastFatherNameENG"),
            customer.MOTHER_NAME.cast("string").alias("FirstMotherNameGR"),
            lit(None).cast("string").alias("LastMotherNameGR"),
            lit(None).cast("string").alias("LastMotherNameENG"),
            customer.NAME_STANDARD.cast("string").alias("LegalNameGR"),
            lit(None).cast("string").alias("LegalNameENG"),
            customer.NAME_STANDARD.cast("string").alias("PrimaryCommercialGR"),
            lit(None).cast("string").alias("PrimaryCommercialENG"),
            customer.TITLE.cast("string").alias("SalutationTypeCode"),
            customer.TITLE.cast("string").alias("SalutationType"),
            customer.SEX.cast("string").alias("GenderCode"),
            party_profits_df.Gender.cast("string").alias("Gender"),
            maritalstatuscode_code.MaritalStatusCode.cast("string").alias(
                "MaritalStatusCode"
            ),
            party_profits_df.MaritalStatus.cast("string").alias("MaritalStatus"),
            citizenship_code.Citizenship.cast("string").alias("Citizenship"),
            citizenship_code.CitizenshipCode.cast("string").alias("CitizenshipCode"),
            nationalitycode_code.NationalityCode.cast("string").alias(
                "NationalityCode"
            ),
            nationality_code.Nationality.cast("string").alias("Nationality"),
            occupationcode_code.OccupationCode.cast("string").alias("OccupationCode"),
            occupation_code.Occupation.alias("Occupation"),
            party_profits_df.BirthDate.cast("string").alias("BirthDate"),
            lit(None).cast("string").alias("BirthPlace"),
            lit(None).cast("date").alias("DeathDate"),
            communicationlanguage_code.CommunicationLanguage.cast("string").alias(
                "CommunicationLanguage"
            ),
            communicationlanguage_code.CommunicationLanguageCode.cast("int").alias(
                "CommunicationLanguageCode"
            ),
            customer.TAX_NUMBER.cast("string").alias("TaxRegistrationNumber"),
            lit(None).cast("string").alias("TaxResidenceCountry"),
            organizationorigincountry_code.OrganizationOriginCountry.cast("string"),
            lit(None).cast("string").alias("PassportNumber"),
            lit(None).cast("string").alias("PassportIssuingAuthority"),
            lit(None).cast("date").alias("PassportIssueDate"),
            lit(None).cast("date").alias("PassportExpiredDate"),
            customer.ID_NO.cast("string").alias("NationalNumber"),
            lit(None).cast("string").alias("NationalNumberIssuingAuthority"),
            party_profits_df.NationalNumberIssueDate.cast("date").alias(
                "NationalNumberIssueDate"
            ),
            lit(None).cast("date").alias("NationalNumberExpiredDate"),
            lit(None).cast("string").alias("MilitaryNumber"),
            lit(None).cast("string").alias("MilitaryAuthority"),
            lit(None).cast("date").alias("MilitaryIssueDate"),
            lit(None).cast("date").alias("MilitaryExpiredDate"),
            lit(None).cast("string").alias("SocialSecurityNumber"),
            lit(None).cast("string").alias("ChildrenCertification"),
            lit(None).cast("string").alias("ChildrenCertificationIssuingAuthority"),
            lit(None).cast("string").alias("EnterpriseStartDate"),
            customer.EXPIRE_DATE.cast("date").alias("EnterpriseEndDate"),
            economicactivitycode_code.EconomicActivityCode.cast("int").alias(
                "EconomicActivityCode"
            ),
            economicactivity_code.EconomicActivity.cast("string").alias(
                "EconomicActivity"
            ),
            branchactivitycode_code.BranchActivityCode.cast("int").alias(
                "BranchActivityCode"
            ),
            BranchActivity_code.BranchActivity.cast("string").alias("BranchActivity"),
            lit(None).cast("string").alias("NumberOfChildren"),
            lit(None).cast("string").alias("GeneralCommercialRegister"),
            lit(None).cast("string").alias("NumberofEmployees"),
            lit(None).cast("string").alias("TotalIncome"),
            lit(None).cast("string").alias("BalanceSheetTotalAmount"),
            lit(None).cast("string").alias("AnnualTurnoverAmount"),
            party_profits_df.AB_GROUP.cast("string").alias("GroupCode"),
            party_profits_df.Group_Description.cast("string").alias("GroupName"),
            party_profits_df.Profits_Group_id.alias("GroupCodePortfolio"),
            lit(None).cast("int").alias("PartyBankruptcyStage"),
            lit(None).cast("string").alias("CovidFlag"),
            lit(None).cast("int").alias("CovidCategoryCode"),
            lit(None).cast("string").alias("CovidCategory"),
            lit(None).alias("AffectedCategoryCode"),
            lit(None).alias("AffectedCategory"),
            customer.gd_CRRAT.cast("string").alias("MonitoringUnitCode"),
            customer.gd_CRRAT.cast("string").alias("MonitoringUnit"),
            customer.gd_BHFMR.cast("int").alias("CreditRatingTypeCode"),
            lit(None).cast("string").alias("CreditRatingType"),
            educationlevelcode_code.EducationLevelCode.cast("int").alias(
                "EducationLevelCode"
            ),
            educationlevel_code.EducationLevel.cast("string").alias("EducationLevel"),
            lit(None).cast("string").alias("BalanceCategoryCode"),
            lit(None).cast("string").alias("BalanceCategory"),
            lit(None).cast("string").alias("ExtraSegmentCode"),
            lit(None).cast("string").alias("ExtraSegment"),
            lit(None).cast("string").alias("SegmentCode"),
            lit(None).cast("string").alias("Segment"),
            lit(None).cast("string").alias("SubSegmentCode"),
            lit(None).cast("string").alias("SubSegment"),
            legalformcode_code.LegalFormCode.cast("int").alias("LegalFormCode"),
            legalform_code.LegalForm.cast("string").alias("LegalForm"),
            lit(None).alias("IsWholeshaleFlag"),
            employmentstatus_code.EmploymentStatus.cast("string").alias(
                "EmploymentStatus"
            ),
            employmentstatus_code.EmploymentStatusCode.cast("int").alias(
                "EmploymentStatusCode"
            ),
            lit(None).cast("date").alias("EmploymentStatusDate"),
            lit(None).cast("string").alias("CreatedBy"),
            lit(None).cast("timestamp").alias("CreatedByDate"),
            lit(None).cast("string").alias("LastUpdatedBy"),
            lit(None).cast("timestamp").alias("LastUpdatedByDate"),
            lit(None).alias("ResubmissionStatus"),
            lit(None).alias("N4605Flag"),
            lit(None).alias("ConnectedBorrowerFlag"),
            lit(None).alias("StaffFlag"),
            lit(None).alias("L3869BorrowerHasOver50KBalance"),
            lit(None).alias("TargetLegalActionCode"),
            lit(None).alias("TargetLegalAction"),
            lit(None).cast("string").alias("SubServicer"),
            party_profits_df.RiskDescription.cast("String").alias("RiskDescription"),
            customer.MOBILE_TEL.cast("string").alias("MobileTel"),
            customer.MOBILE_TEL2.cast("string").alias("MobileTel2"),
            customer.businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        party_profits_df.unpersist()

        party_profits_df_final.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_party_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)